In [ ]:
import pandas as pd
from datetime import date

from nba_ml.stores import player_games
from nba_ml.stores import players, teams
from nba_ml.feature_engineering import game_columns as processing
from nba_ml import season
from nba_ml.entities import Game
from nba_ml.models.player_stats import features as player_stats

from nba_ml.stores import team_games

In [ ]:
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 250)

# PLAYER GAME DATA: #

In [ ]:
player_game_data = player_games.get_all_player_game_data()
player_game_data = player_game_data.dropna()
player_game_data

In [ ]:
player_game_data = processing.add_points_column(
    player_game_data,
    player_games.FIELD_GOALS_MADE.category_name, 
    player_games.THREE_POINTERS_MADE.category_name, 
    player_games.FREE_THROWS_MADE.category_name
    )
player_game_data

In [ ]:
player_game_data = player_game_data.sort_values(by=player_games.GAME_DATE.category_name)
player_game_data

In [ ]:
player_game_data =processing.add_break_columns(
    player_game_data,
    player_games.GAME_DATE.category_name
    )
player_game_data

In [ ]:
player_game_data = processing.add_total_rebounds_column(
    player_game_data,
    player_games.OFFENSIVE_REBOUNDS.category_name,
    player_games.DEFENSIVE_REBOUNDS.category_name    
    )
player_game_data

In [ ]:
new_dict = {}
for player_id, game_data in player_game_data.groupby(by=player_games.PLAYER_ID.category_name):
    sorted_data = game_data.sort_values(by=player_games.GAME_DATE.category_name)
    new_dict[player_id] = processing.add_break_columns(
    sorted_data,
    player_games.GAME_DATE.category_name
    )

new_dict

In [ ]:
player_games.get_game_data_after_date(datetime(2025, 6, 1))

# TEAM GAME DATA: #

In [ ]:
team_game_data = team_games.get_all_team_game_data()
team_game_data = team_game_data.dropna()
team_game_data

In [ ]:
team_game_data = team_game_data.sort_values(by=team_games.GAME_DATE.category_name)
team_game_data

In [ ]:
team_game_data = processing.add_encoded_home_or_away_column(
    team_game_data,
    team_games.MATCHUP.category_name
    )
team_game_data

In [ ]:
team_game_data = processing.add_possessions_column(
    team_game_data,
    team_games.FIELD_GOALS_ATTEMPTED.category_name,
    team_games.FREE_THROWS_ATTEMPTED.category_name,
    team_games.OFFENSIVE_REBOUNDS.category_name,
    team_games.TURNOVERS.category_name    
    )
team_game_data

In [ ]:
team_game_data = processing.add_encoded_season_type_columns(
    team_game_data,
    team_games.GAME_ID.category_name
    )
team_game_data

In [ ]:
feature_set = player_stats.build_feature_set([player_stats.TargetStat.POINTS])

team_game_data = processing.add_opponent_stat_columns(
    team_game_data,
    team_games.GAME_ID.category_name,
    team_games.TEAM_ID.category_name,
    feature_set.opponent_columns_to_copy
    )
team_game_data

# PLAYER INFO: #

In [ ]:
player_info = players.get_all_players_info()
player_info = player_info.dropna()
player_info

In [ ]:
player_names = {player.id: player.name for player in source.players()}
player_info['NAME'] = player_info[players.PLAYER_ID.category_name].map(player_names)
player_info

In [ ]:
na_data = player_info[player_info[players.HEIGHT.category_name].isna()]

In [ ]:
missing_data = player_info[
    (player_info[players.HEIGHT.category_name].astype(str) == '') |
    (player_info[players.WEIGHT.category_name].astype(str) == '') |
    (player_info[players.POSITION.category_name].astype(str) == '') | 
    (player_info[players.FROM_YEAR.category_name].astype(str) == '') | 
    (player_info[players.TO_YEAR.category_name].astype(str) == '')
]
missing_data

In [ ]:
missing_player_ids = set(missing_data[players.PLAYER_ID.category_name])

games_with_missing_player_info = player_game_data[
    player_game_data[player_games.PLAYER_ID.category_name].isin(
        missing_player_ids
    )
    ]

games_with_missing_player_info

In [ ]:
invalid_weights = player_info[
    player_info[players.WEIGHT.category_name].astype(str) == ''
]
invalid_weights

In [ ]:
player_info = processing.convert_height_to_inches(
    player_info,
    players.HEIGHT.category_name
)
player_info

In [ ]:
positions = set(player_info[players.POSITION.category_name])
positions

In [ ]:
player_info[players.BIRTHDATE.category_name] = pd.to_datetime(player_info[players.BIRTHDATE.category_name])
player_info

In [ ]:
player_info[players.FROM_YEAR.category_name] = pd.to_datetime(player_info[players.FROM_YEAR.category_name], format='%Y')
player_info

In [ ]:
player_game_data_with_info = player_game_data.merge(
    player_info, 
    left_on=player_games.PLAYER_ID.category_name, 
    right_on=players.PLAYER_ID.category_name
    )
player_game_data_with_info

# NBA DATA PROCESSING #

In [ ]:
player_team_info_game_data = processing.merge_player_team_game_data(
        player_game_data_with_info,
        team_game_data,
        team_games.GAME_ID.category_name,
        player_games.GAME_ID.category_name,
        team_games.TEAM_ID.category_name,
        player_games.TEAM_ID.category_name
    )
player_team_info_game_data

In [ ]:
preprocessed_data = player_stats.build_histories(
    feature_set,
    player_info,
    player_game_data,
    team_game_data,
    True
)
preprocessed_data

# PLAYER PROPS: #

In [ ]:
sequences, training_samples, validation_samples, test_samples = player_stats.get_all_labeled_data(
        feature_set,
        validation_split_start_date = datetime(2024, 1, 1),
        test_split_start_date = datetime(2025, 1, 1),
        chunk_step_size = 250,
        print_debug = True
        )

In [ ]:
PLAYER_NUM = 516

In [ ]:
sequences.sequences[PLAYER_NUM]

In [ ]:
datasets[0][PLAYER_NUM]['PAST_GAMES_AGAINST_OPPONENT_SEQUENCE']

In [ ]:
datasets[0][PLAYER_NUM]['STATIC_DATA']

In [ ]:
datasets[0][PLAYER_NUM]['PLAYER_INFO']

In [ ]:
datasets[1][PLAYER_NUM]

In [ ]:
removed_core_teammates: set[int] = set([
    players.find_by_name('Josh Hart')[0].id
])

added_core_teammates: set[int] = set([

])

target_game = Game(
    date(2026, 6, 30),
    teams.find('Knicks').id,
    teams.find('Spurs').id,
    season.SeasonType.PLAYOFFS
)

input_data = player_stats.get_inference_data(
    feature_set,
    players.find_by_name('Jalen Brunson')[0].id,
    target_game.home_team_id,
    target_game,
    removed_core_teammates,
    added_core_teammates,
    True
)

In [ ]:
input_data